# 🚀 StyleMatch AI - Colab Runner

### Hướng dẫn:
1. Bật GPU: `Runtime` → `Change runtime type` → **T4 GPU**
2. Chạy tuần tự Cell 1 → Cell 5
3. Cell 5 chạy liên tục: health 30s, search log 2s → Discord
4. Bấm **■ Stop** để dừng

## 📁 Cell 1: Clone source code

In [ ]:
!git clone https://github.com/Muhamad-Shahan/Stylematch-ai.git
%cd Stylematch-ai

import os
required = ['app.py', 'articles.csv', 'embeddings.npy', 'ids.npy']
for f in required:
    s = '✅' if os.path.exists(f) else '❌ THIẾU'
    print(f'{s} {f}')
print(f"{'✅' if os.path.isdir('images') else '❌ THIẾU'} images/")

## 📦 Cell 2: Cài thư viện

In [ ]:
%pip install -q streamlit pyngrok transformers torch flask pandas numpy Pillow

## 🔑 Cell 3: Ngrok Token
Lấy tại: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_TOKEN = "YOUR_TOKEN"
!ngrok config add-authtoken {NGROK_TOKEN}

## 🧠 Cell 4: Load Model & Start Backend

In [ ]:
import numpy as np
import pandas as pd
import torch
import time as _time
from transformers import CLIPProcessor, CLIPModel
from flask import Flask, request, jsonify
from PIL import Image
import io, threading, logging
from datetime import datetime

print("⏳ Đang tải CLIP Model...")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print(f"✅ CLIP Model trên: {device.upper()}")

print("⏳ Đang tải database...")
embeddings = np.load("embeddings.npy").astype(np.float32)
ids = np.load("ids.npy", allow_pickle=True)
df = pd.read_csv("articles.csv", dtype={'article_id': str})
print(f"✅ Database: {len(ids)} items")

search_logs = []
total_searches = 0

logging.getLogger('werkzeug').setLevel(logging.ERROR)
flask_app = Flask(__name__)

@flask_app.route('/health')
def health():
    return jsonify({"status":"ok","device":device,"items":len(ids),"total_searches":total_searches})

@flask_app.route('/logs')
def get_logs():
    return jsonify(search_logs)

@flask_app.route('/search', methods=['POST'])
def search():
    global total_searches
    start = _time.time()
    file = request.files['image']
    image = Image.open(io.BytesIO(file.read())).convert('RGB')
    top_k = int(request.form.get('top_k', 5))

    inputs = processor(images=image, return_tensors='pt', padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        qv = model.get_image_features(**inputs)
        if not isinstance(qv, torch.Tensor):
            qv = qv.pooler_output if hasattr(qv,'pooler_output') else qv[1]
            if hasattr(model,'visual_projection') and qv.shape[-1]!=512:
                qv = model.visual_projection(qv)
        qv = qv / qv.norm(p=2, dim=-1, keepdim=True)
        qv = qv.cpu().numpy()

    scores = np.dot(qv, embeddings.T).flatten()
    top_idx = np.argsort(scores)[-top_k:][::-1]

    results = []
    for idx in top_idx:
        rid = str(ids[idx])
        meta = df[df['article_id']==rid]
        name = meta.iloc[0].get('prod_name','Item') if len(meta)>0 else 'Item'
        results.append({'index':int(idx),'id':rid,'score':float(scores[idx]),'name':name})

    elapsed = round(_time.time()-start,3)
    total_searches += 1
    search_logs.append({
        'time':datetime.now().strftime('%H:%M:%S'),
        'search_no':total_searches,
        'top_result':results[0]['name'] if results else '?',
        'top_score':round(results[0]['score']*100,1) if results else 0,
        'elapsed_ms':int(elapsed*1000)
    })
    return jsonify(results)

threading.Thread(target=lambda: flask_app.run(host='0.0.0.0',port=5001,debug=False,use_reloader=False), daemon=True).start()
_time.sleep(2)
print()
print('='*60)
print('🚀 BACKEND ĐANG CHẠY: http://localhost:5001')
print('='*60)
print('\n👉 Chạy Cell 5!')

## 🌐 Cell 5: Streamlit + Ngrok + Monitor → Discord

In [ ]:
from pyngrok import ngrok
import time, requests, threading

DISCORD_WEBHOOK = "https://discord.com/api/webhooks/1491031437288280146/B5xnLJ_qxU_nxL63GnzUmSvhCHGMw9Dxv5SzxQLyBJVs0V-Y2Jf22y2svv6DLiqPRti5"

def send_discord(content=None, embed=None):
    payload = {}
    if content: payload['content'] = content
    if embed: payload['embeds'] = [embed]
    try: requests.post(DISCORD_WEBHOOK, json=payload, timeout=5)
    except: pass

print('⏳ Kiểm tra Backend...')
try:
    r = requests.get('http://localhost:5001/health', timeout=5)
    info = r.json()
    print(f"✅ Backend OK! {info['device'].upper()} | {info['items']} items")
except:
    print('❌ Backend chưa chạy! Chạy Cell 4 trước.')
    raise SystemExit

ngrok.kill()
print('⏳ Khởi động Streamlit...')
get_ipython().system_raw('nohup streamlit run app.py --server.port 8501 --server.headless true &')
time.sleep(5)

try:
    public_url = ngrok.connect(8501).public_url
except Exception as e:
    print(f'❌ Ngrok: {e}')
    raise SystemExit

print()
print('='*60)
print(f'🌍 APP: {public_url}')
print('='*60)

send_discord(embed={
    'title':'🚀 StyleMatch AI — Started',
    'color':0x2ECC71,
    'fields':[
        {'name':'🌐 URL','value':public_url,'inline':False},
        {'name':'🖥️ Device','value':info['device'].upper(),'inline':True},
        {'name':'📦 Items','value':str(info['items']),'inline':True}
    ]
})
print('✅ Đã gửi Discord!')
print()
print('📡 Monitor chạy... (■ Stop để dừng)')
print('-'*60)

def health_loop():
    while True:
        now = time.strftime('%H:%M:%S')
        try:
            r = requests.get('http://localhost:5001/health', timeout=5)
            i = r.json()
            msg = f"[{now}] ❤️ Health OK | {i['device'].upper()} | {i['items']} items | Searches: {i['total_searches']}"
            print(msg)
            send_discord(content=f'```{msg}```')
        except:
            msg = f'[{now}] ⚠️ Backend không phản hồi!'
            print(msg)
            send_discord(content=f'```{msg}```')
        time.sleep(30)

threading.Thread(target=health_loop, daemon=True).start()

last_log_count = 0
try:
    while True:
        try:
            r = requests.get('http://localhost:5001/logs', timeout=3)
            logs = r.json()
            for entry in logs[last_log_count:]:
                t = entry.get('time',time.strftime('%H:%M:%S'))
                msg = f"[{t}] 🔍 Search #{entry['search_no']} | Top: {entry['top_result']} ({entry['top_score']}%) | {entry['elapsed_ms']}ms"
                print(msg)
                send_discord(content=msg)
            last_log_count = len(logs)
        except: pass
        time.sleep(2)
except KeyboardInterrupt:
    print()
    print('='*60)
    print('🛑 Monitor đã dừng.')
    print('='*60)
    send_discord(embed={'title':'🛑 StyleMatch AI — Stopped','color':0xE74C3C,'description':'Monitor đã dừng.'})